In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import joblib

# 1. LOAD: Read your Kaggle file
df = pd.read_csv('global_fuel_prices_2020_2026.csv')

# 2. CLEAN COLUMN NAMES & DATES
df.columns = df.columns.str.strip().str.lower()
df['date'] = pd.to_datetime(df['date'])

# 3. SELECT & AGGREGATE
# Since there are multiple countries, we take the average price per day for our global AI model
# Targets: diesel_usd_liter (Primary), brent_crude_usd (Market Indicator)
targets = ['diesel_usd_liter', 'petrol_usd_liter', 'brent_crude_usd']

print("Aggregating data by date...")
# Grouping by date ensures we have one row per day for the LSTM time-series
data_grouped = df.groupby('date')[targets].mean().sort_index()

# 4. HANDLE MISSING
data_grouped = data_grouped.ffill().bfill() 

# 5. NORMALIZE: Squash values for the LSTM model (Aaryan's task)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data_grouped)

# 6. SAVE: Create the files for Step 2
# Save the cleaned CSV
processed_df = pd.DataFrame(scaled_data, columns=targets, index=data_grouped.index)
processed_df.to_csv('cleaned_fuel_prices.csv')

# Save the scaler so we can "un-scale" prices for the dashboard later
joblib.dump(scaler, 'scaler.pkl')

print(f"Step 1 Complete! Data grouped by date. Rows processed: {len(processed_df)}")
print("Columns ready for AI: ", targets)

FileNotFoundError: [Errno 2] No such file or directory: 'global_fuel_prices_2020_2026.csv'